In [ ]:
## Investment Universe Construction
Select firms that are present in the cleaned returns and emissions datasets.

In [12]:
import numpy as np

In [1]:
import pandas as pd
from pathlib import Path

BASE_DIR = Path.cwd().parent
CLEAN_DIR = BASE_DIR / "data_clean"

# Load the cleaned datasets
returns_non_stale = pd.read_excel(CLEAN_DIR / "returns_monthly_non_stale.xlsx")
scope1_clean = pd.read_excel(CLEAN_DIR / "scope1_clean.xlsx")
scope2_clean = pd.read_excel(CLEAN_DIR / "scope2_clean.xlsx")
revenues_clean = pd.read_excel(CLEAN_DIR / "revenues_clean.xlsx")
mv_year_clean = pd.read_excel(CLEAN_DIR / "market_value_year_clean.xlsx")
static = pd.read_excel(BASE_DIR / "data_raw" / "Static_2025.xlsx")

In [3]:
# Filter firms in Europe (EUR) from the static dataset
static_europe = static[static["Region"].str.upper() == "EUR"].copy()

# Check the number of firms in Europe
print(f"Firms in Europe: {static_europe.shape[0]}")

Firms in Europe: 633


In [4]:
# Get the ISINs for each dataset
isins_europe = set(static_europe["ISIN"])
isins_returns = set(returns_non_stale["ISIN"])
isins_scope1 = set(scope1_clean["ISIN"])
isins_scope2 = set(scope2_clean["ISIN"])
isins_revenues = set(revenues_clean["ISIN"])
isins_mv_year = set(mv_year_clean["ISIN"])

# Find the common ISINs across all datasets
common_isins = (
    isins_europe
    & isins_returns
    & isins_scope1
    & isins_scope2
    & isins_revenues
    & isins_mv_year
)

# Output the number of firms in the final universe
print(f"Number of firms in the final investment universe: {len(common_isins)}")

Number of firms in the final investment universe: 616


In [5]:
# Filter the datasets based on the common ISINs
returns_final = returns_non_stale[returns_non_stale["ISIN"].isin(common_isins)].copy()
scope1_final = scope1_clean[scope1_clean["ISIN"].isin(common_isins)].copy()
scope2_final = scope2_clean[scope2_clean["ISIN"].isin(common_isins)].copy()
revenues_final = revenues_clean[revenues_clean["ISIN"].isin(common_isins)].copy()
mv_year_final = mv_year_clean[mv_year_clean["ISIN"].isin(common_isins)].copy()
static_final = static_europe[static_europe["ISIN"].isin(common_isins)].copy()

# Check the shape of the filtered datasets
print(f"Returns final shape: {returns_final.shape}")
print(f"Scope1 final shape: {scope1_final.shape}")
print(f"Scope2 final shape: {scope2_final.shape}")
print(f"Revenues final shape: {revenues_final.shape}")
print(f"Market value final shape: {mv_year_final.shape}")
print(f"Static final shape: {static_final.shape}")

Returns final shape: (616, 316)
Scope1 final shape: (616, 29)
Scope2 final shape: (616, 29)
Revenues final shape: (616, 29)
Market value final shape: (616, 29)
Static final shape: (616, 4)


In [ ]:
## Calculate expected returns

In [8]:
# Remove non-numeric columns (NAME, ISIN)
returns_matrix = returns_final.drop(columns=["NAME", "ISIN"])

# Check the result
returns_matrix.head()

,1999-12-31,2000-01-31,2000-02-29,2000-03-31,2000-04-28,2000-05-31,2000-06-30,2000-07-31,2000-08-31,2000-09-29,...,2025-04-30,2025-05-30,2025-06-30,2025-07-31,2025-08-29,2025-09-30,2025-10-31,2025-11-28,2025-12-31,2026-01-30
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.230679,0.001302,0.112875,-0.004401,-0.025746,-0.007655,-0.140520,0.141737,0.063197,0.094270
5,NaN,0.057243,-0.015488,0.031138,-0.134598,0.085088,0.072038,-0.053858,0.015025,0.054059,...,0.040736,0.013612,0.045520,-0.024938,0.007323,-0.003770,-0.006460,0.051208,0.023005,-0.001547
6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.041736,0.143363,-0.004486,-0.041492,0.140807,0.037803,0.082711,0.086321,0.109326,0.128831
7,NaN,-0.078007,0.029818,0.028238,-0.043483,0.017996,0.042918,0.003625,0.042859,-0.042221,...,-0.018134,0.241551,0.055130,0.086324,0.032162,0.027326,0.059580,0.054800,0.106051,0.077952
8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.135199,0.045852,0.066444,-0.060294,0.029221,-0.024019,-0.003705,-0.028282,0.021079,0.009623


In [9]:
# Calculate the expected return (mean) for each firm (monthly returns)
mu = returns_matrix.mean(axis=1)

# Check the result (expected returns per firm)
mu.head()

4    0.011228
5    0.012403
6    0.010545
7    0.016956
8    0.007529
dtype: float64